# Impulsive noise removal and deconvolution in RGB images

A Gaussian blur is applied separately to the R, G, and B channels of an RGB image. 
Impulsive corruption acts on **whole RGB pixels**: when a spatial position is selected, all three channel values are replaced by independent outliers.

For an RGB residual $r_{ij}\in\mathbb R^3$, the two robust data terms are

$$
S_{\rm HuberL2}(r)=\sum_{ij} w_{ij}\,\rho_\sigma(\|r_{ij}\|_2),
\qquad
S_{\rm L1L2}(r)=\sum_{ij} w_{ij}\,\|r_{ij}\|_2.
$$

Both couple the three colors **within each pixel** (rather than treating the channels as unrelated scalar observations). The spatial weights $w_{ij}$ are supplied by the RegPy grid.

We compare an ordinary quadratic fidelity, HuberL2 with and without bilateral constraints, and L1L2 with bilateral constraints. 

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from regpy.operators import GaussianBlur
from regpy.vecsps import UniformGridFcts
from regpy.solvers import Setting
from regpy.solvers.linear import TikhonovCG
from regpy.solvers.plotting import convergence_plots
from regpy.hilbert import L2
from regpy.functionals import QuadBil
# HuberL2 exists in this module but is not listed in its __all__.
# Use explicit imports rather than `from regpy.functionals import *`.
from regpy.functionals.vector_integral_functionals import HuberL2, L1L2

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

## 1. Load a color image and set up the vector-valued grid

Set `IMAGE_PATH` to your own RGB image (`.png`, `.jpg`, `.tif`, ...).  
`MAX_SIDE` controls memory and runtime; increase it for a full-resolution experiment.

In [ ]:
IMAGE_PATH = Path('images/peppers.png')
MAX_SIDE = 192               # set to 512 or None to retain more resolution
RANDOM_SEED = 20261013

with Image.open(IMAGE_PATH) as image:
    image = image.convert('RGB')  # discard alpha channel, if present
    if MAX_SIDE is not None:
        image.thumbnail((MAX_SIDE, MAX_SIDE), Image.Resampling.LANCZOS)
    exact_sol = np.asarray(image, dtype=np.float64) / 255.0

height, width, channels = exact_sol.shape
grid = UniformGridFcts((0, 1, height), (0, 1, width), shape_codomain=(3,))
print(f'RGB image: {exact_sol.shape}; RegPy grid: {grid.shape}')
plt.figure(figsize=(6, 5))
plt.imshow(exact_sol)
plt.title('Exact RGB image')
plt.axis('off');

## 2. Blur and corrupt entire RGB pixels

The **same Boolean pixel mask** applies to R, G and B. Each damaged pixel receives a random 3-vector, not a single damaged channel. The outliers are deliberately outside $[0,1]$; clipping is used **only for visualization**, never on the observed data given to the solvers.

In [ ]:
BLUR_SIGMA = 0.025
PAD_AMOUNT = 12
CORRUPTION_FRACTION = 0.04
OUTLIER_AMPLITUDE = 8.0

conv = GaussianBlur(grid, BLUR_SIGMA, pad_amount=PAD_AMOUNT)
blur = np.maximum(conv(exact_sol), 0.0)

rng = np.random.default_rng(RANDOM_SEED)
n_bad = max(1, round(CORRUPTION_FRACTION * height * width))
corruption_mask = np.zeros((height, width), dtype=bool)
chosen_pixels = rng.choice(height * width, size=n_bad, replace=False)
corruption_mask.flat[chosen_pixels] = True

data = blur.copy()
# Boolean indexing produces an (n_bad, 3) view of selected RGB vectors.
# Each selected spatial pixel is replaced in ALL three components.
data[corruption_mask, :] = rng.uniform(
    -OUTLIER_AMPLITUDE, OUTLIER_AMPLITUDE, size=(n_bad, 3)
)

assert np.array_equal(data[~corruption_mask], blur[~corruption_mask])
assert np.isfinite(data).all()
print(f'Corrupted pixels: {n_bad} / {height * width} '
      f'({100*n_bad/(height*width):.2f}%). Each affected pixel has 3 new values.')
print(f'Data range before display clipping: [{data.min():.3g}, {data.max():.3g}]')

### RGB-aware plots

Displaying a residual RGB array as if it were an image is misleading because residuals can be negative. We show the observed image with display-only clipping and the **Euclidean magnitude of its RGB residual** as a heatmap.

In [ ]:
def show_rgb_comparison(reference, candidate, *, title='Reconstruction',
                        candidate_title='Reconstruction', mask=None):
    """Compare two (H,W,3) arrays and show per-pixel RGB residual magnitudes."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), constrained_layout=True)
    axes[0].imshow(np.clip(reference, 0.0, 1.0))
    axes[0].set_title('Ground truth')
    axes[1].imshow(np.clip(candidate, 0.0, 1.0))
    axes[1].set_title(candidate_title + ' (display clipped)')
    error = np.linalg.norm(candidate - reference, axis=-1)
    im = axes[2].imshow(error, cmap='magma')
    axes[2].set_title('RGB error magnitude per pixel')
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
    if mask is not None:
        # Show the selected damaged pixel locations without changing the data.
        axes[1].contour(mask.astype(float), levels=[0.5], colors='cyan', linewidths=0.35)
    for ax in axes:
        ax.set_axis_off()
    fig.suptitle(title)
    return fig


def rgb_relative_l2(x, reference):
    return np.linalg.norm(np.ravel(x - reference)) / max(
        np.linalg.norm(np.ravel(reference)), 1e-15
    )


def show_rgb_reconstructions(setting, reconstructions, reference, *, title):
    """Display ground truth and multiple reconstructions without transposing RGB."""
    names = ['Ground truth'] + list(reconstructions)
    images = [reference] + list(reconstructions.values())
    ncols = 3
    nrows = (len(names) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.2*nrows),
                             squeeze=False, constrained_layout=True)
    for ax, name, img in zip(axes.flat, names, images):
        ax.imshow(np.clip(img, 0.0, 1.0))
        if name != 'Ground truth':
            rule = setting.get_stoprule(name).main_rule
            status = ('converged' if rule.converged is True else
                      'not converged' if rule.converged is False else
                      'unconfirmed')
            name = f'{name} ({status})\nrel. RGB L2 err.: {rgb_relative_l2(img, reference):.3g}'
        ax.set_title(name, fontsize=10)
        ax.axis('off')
    for ax in list(axes.flat)[len(names):]:
        ax.axis('off')
    fig.suptitle(title)
    return fig


show_rgb_comparison(exact_sol, data, title='Vector-valued impulsive noise',
                    candidate_title='Observed RGB data', mask=corruption_mask);

## 3. Quadratic data fidelity

As in the grayscale example, least-squares fidelity is sensitive to large impulsive errors. The regularization is $R(f)=\|f\|_{L^2}^2$ (with the normalization used internally by RegPy).

In [ ]:
ALPHA = 1e-3
quadratic_setting = Setting(conv, L2, L2, data=data, regpar=ALPHA)
quadratic_solver = TikhonovCG(quadratic_setting, reltolx=1e-6)
quadratic_reco, _ = quadratic_solver.run()
show_rgb_comparison(exact_sol, quadratic_reco,
                    title='Quadratic data fidelity: sensitivity to outliers',
                    candidate_title='Quadratic Tikhonov reconstruction');

## 4. HuberL2 data fidelity, quadratic penalty

The Huber transition is applied to the **norm of the three-component residual at each pixel**, not separately to each channel. The implementation `HuberL2` uses the scalar Huber functional with its built-in normalization (see your `vector_integral_functionals.py`).

In [ ]:
HUBER_SIGMA = 1e-4
ALPHA = 5e-3
huber_setting = Setting(
    conv, L2, HuberL2(grid, sigma=HUBER_SIGMA),
    data=data, regpar=ALPHA
)
huber_setting.display_all_methods()

In [ ]:
MAX_OP_EVALS = 8000


def run_methods(setting, method_names, *, max_op_evals=MAX_OP_EVALS):
    reconstructions = {}
    for method in method_names:
        try:
            x, _ = setting.run(method,  max_it = 3500,max_op_evals=max_op_evals)
        except Exception as exc:
            print(f'{method}: FAILED ({type(exc).__name__}: {exc})')
            continue
        reconstructions[method] = np.array(x, copy=True)
        print(f'{method:8s}: RGB relative L2 reconstruction error = '
              f'{rgb_relative_l2(x, exact_sol):.5g}')
    return reconstructions


huber_methods = ['FISTA', 'dFISTA', 'PDHG']#, 'dPDHG', 'dFB']
huber_recos = run_methods(huber_setting, huber_methods)
convergence_plots(huber_setting, methods=list(huber_recos),
                  title='HuberL2 fidelity, quadratic penalty')
show_rgb_reconstructions(huber_setting, huber_recos, exact_sol,
                         title='HuberL2 / unconstrained L2');

## 5. L1L2 fidelity

The last case uses the pixelwise isotropic $\ell^1(\ell^2)$ data fidelity

$$ S(r) = \sum_{ij} w_{ij}\sqrt{r_{ij,R}^2+r_{ij,G}^2+r_{ij,B}^2}. $$

The three channels share a single robust penalty at each pixel; this is suited to impulse noise that damages RGB pixel triples.

In [ ]:
l1l2_setting = Setting(
    conv, L2, L1L2(grid), data=data, regpar=ALPHA
)
l1l2_setting.display_all_methods()
l1l2_methods = ['dFISTA', 'PDHG', 'dPDHG']
l1l2_recos = run_methods(l1l2_setting, l1l2_methods)
convergence_plots(l1l2_setting, methods=list(l1l2_recos),
                  title='L1L2 fidelity, constrained quadratic penalty')
show_rgb_reconstructions(l1l2_setting, l1l2_recos, exact_sol,
                         title='L1L2 / bilateral RGB constraints');